In [1]:
import pandas as pd
import numpy as np
from blackscholes import *

import warnings
warnings.filterwarnings('ignore')

In [2]:
SnP_df = pd.read_csv('/Users/artyomkraevskiy/Desktop/RPD paper/Data/Options_pricing/SnP_data.csv')
SnP_df['Date'] = pd.to_datetime(SnP_df['Date'])

VIX_df = pd.read_csv('/Users/artyomkraevskiy/Desktop/RPD paper/Data/Options_pricing/VIX_data.csv')
VIX_df.rename(columns = {'DATE' : 'Date', 
                          'OPEN' : 'VIX'}, inplace = True)
VIX_df = VIX_df[['Date', 'VIX']]
VIX_df['Date'] = pd.to_datetime(VIX_df['Date'])

rf_rate = pd.read_csv('/Users/artyomkraevskiy/Desktop/RPD paper/Data/Options_pricing/risk_free_gov_bonds_rate.csv')
rf_rate.rename(columns = {
    '1 Mo' : 1,
    '2 Mo' : 2,
    '3 Mo' : 3,
    '4 Mo' : 4,
    '6 Mo' : 6,
    '1 Yr' : 12,
    '2 Yr' : 24,
    '3 Yr' : 36,
    '5 Yr' : 60,
    '7 Yr' : 84,
    '10 Yr' : 120,
    '20 Yr' : 240,
    '30 Yr' : 360
}, inplace = True)

rf_rate['Date'] = pd.to_datetime(rf_rate['Date'])
rf_rate.fillna(method='ffill', inplace = True)


VIX_df = VIX_df[VIX_df.Date.isin(SnP_df.Date)].reset_index(drop = True)
VIX_df = VIX_df[VIX_df.Date.isin(rf_rate.Date)].reset_index(drop = True)
SnP_df = SnP_df[SnP_df.Date.isin(VIX_df.Date)].reset_index(drop = True)
SnP_df = SnP_df[SnP_df.Date.isin(rf_rate.Date)].reset_index(drop = True)
rf_rate = rf_rate[rf_rate.Date.isin(VIX_df.Date)].reset_index(drop = True)
rf_rate = rf_rate[rf_rate.Date.isin(SnP_df.Date)].reset_index(drop = True)



In [3]:
def find_nearest_rate(possible_vals, a0):

    possible_vals = np.sort(np.array(possible_vals))
    idx = np.abs(possible_vals - a0).argmin()
    first_bound = possible_vals.flat[idx]
    
    try:
        if first_bound < a0:
            second_bound = possible_vals.flat[idx+1]
            return first_bound, second_bound
        else:
            second_bound = possible_vals.flat[idx-1]
            return second_bound, first_bound
    except:
        raise Exception('Index can not be greater than the max value of array')

def get_interpolated_rates_df(rate_df, max_tenor = 361, min_tenor = 1):

    rate_df = rate_df.copy()
    present_rates = rate_df.columns[1:].to_list()

    for r in range(min_tenor, max_tenor):
        if r in present_rates:
            pass
        else:
            bound_1, bound_2 = find_nearest_rate(present_rates, r)
            distance_1, distance_2 = (r - bound_1) / (bound_2 - bound_1), (bound_2 - r) / (bound_2 - bound_1)
            rate_df[r] = distance_1 * rate_df[bound_1] + distance_2 * rate_df[bound_2]

    return rate_df



In [4]:
rf_rate

,Date,1,2,3,4,6,12,24,36,60,84,120,240,360
0,2023-12-29,5.60,5.59,5.40,5.41,5.26,4.79,4.23,4.01,3.84,3.88,3.88,4.20,4.03
1,2023-12-28,5.57,5.55,5.45,5.42,5.28,4.82,4.26,4.02,3.83,3.84,3.84,4.14,3.98
2,2023-12-27,5.55,5.53,5.44,5.42,5.26,4.79,4.20,3.97,3.78,3.81,3.79,4.10,3.95
3,2023-12-26,5.53,5.52,5.45,5.44,5.28,4.83,4.26,4.05,3.89,3.91,3.89,4.20,4.04
4,2023-12-22,5.54,5.52,5.44,5.45,5.31,4.82,4.31,4.04,3.87,3.92,3.90,4.21,4.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8486,1990-01-08,3.67,2.22,7.79,4.32,7.88,7.81,7.90,7.95,7.92,8.05,8.02,6.12,8.09
8487,1990-01-05,3.67,2.22,7.79,4.32,7.85,7.79,7.90,7.94,7.92,8.03,7.99,6.12,8.06
8488,1990-01-04,3.67,2.22,7.84,4.32,7.90,7.82,7.92,7.93,7.91,8.02,7.98,6.12,8.04
8489,1990-01-03,3.67,2.22,7.89,4.32,7.94,7.85,7.94,7.96,7.92,8.04,7.99,6.12,8.04


In [5]:
df_rates = get_interpolated_rates_df(rf_rate)

In [6]:
df_rates

,Date,1,2,3,4,6,12,24,36,60,...,350,351,352,353,354,355,356,357,358,359
0,2023-12-29,5.60,5.59,5.40,5.41,5.26,4.79,4.23,4.01,3.84,...,4.185833,4.18725,4.188667,4.190083,4.1915,4.192917,4.194333,4.19575,4.197167,4.198583
1,2023-12-28,5.57,5.55,5.45,5.42,5.28,4.82,4.26,4.02,3.83,...,4.126667,4.12800,4.129333,4.130667,4.1320,4.133333,4.134667,4.13600,4.137333,4.138667
2,2023-12-27,5.55,5.53,5.44,5.42,5.26,4.79,4.20,3.97,3.78,...,4.087500,4.08875,4.090000,4.091250,4.0925,4.093750,4.095000,4.09625,4.097500,4.098750
3,2023-12-26,5.53,5.52,5.45,5.44,5.28,4.83,4.26,4.05,3.89,...,4.186667,4.18800,4.189333,4.190667,4.1920,4.193333,4.194667,4.19600,4.197333,4.198667
4,2023-12-22,5.54,5.52,5.44,5.45,5.31,4.82,4.31,4.04,3.87,...,4.196667,4.19800,4.199333,4.200667,4.2020,4.203333,4.204667,4.20600,4.207333,4.208667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8486,1990-01-08,3.67,2.22,7.79,4.32,7.88,7.81,7.90,7.95,7.92,...,6.284167,6.26775,6.251333,6.234917,6.2185,6.202083,6.185667,6.16925,6.152833,6.136417
8487,1990-01-05,3.67,2.22,7.79,4.32,7.85,7.79,7.90,7.94,7.92,...,6.281667,6.26550,6.249333,6.233167,6.2170,6.200833,6.184667,6.16850,6.152333,6.136167
8488,1990-01-04,3.67,2.22,7.84,4.32,7.90,7.82,7.92,7.93,7.91,...,6.280000,6.26400,6.248000,6.232000,6.2160,6.200000,6.184000,6.16800,6.152000,6.136000
8489,1990-01-03,3.67,2.22,7.89,4.32,7.94,7.85,7.94,7.96,7.92,...,6.280000,6.26400,6.248000,6.232000,6.2160,6.200000,6.184000,6.16800,6.152000,6.136000


In [7]:
# ============================
# functions for data generation
# ============================

def BlackSholes(S, K, T, r, sigma, type):
    if type == 'call':
        return BlackScholesCall(S=S, K=K, T=T, r=r, sigma=sigma, q=0).price()
    elif type == 'put':
        return BlackScholesPut(S=S, K=K, T=T, r=r, sigma=sigma, q=0).price()
    
def BlackSholesDelta(S, K, T, r, sigma, type):
    if type == 'call':
        return BlackScholesCall(S=S, K=K, T=T, r=r, sigma=sigma, q=0).delta()
    elif type == 'put':
        return BlackScholesPut(S=S, K=K, T=T, r=r, sigma=sigma, q=0).delta()


def get_BS_market_prices(Asset_df, volatility_df, rates_df, strikes, tenors, type = 'vanilla'):

    Asset_df, volatility_df, rates_df  = Asset_df.copy(), volatility_df.copy(), rates_df.copy()
    res_df = Asset_df.copy()
    res_df = res_df.merge(volatility_df, on = 'Date', how = 'left')
    res_df = res_df.merge(rates_df, on = 'Date', how = 'left')

    for t in tenors:
        for s in strikes:
            if type == 'vanilla':
                res_df[f'BS_opt_t{t}_s{s}_call'] = res_df.apply(lambda x: BlackSholes(x['SnP500'], s, t/12, x[t]/100, x['VIX']/100, 'call'), axis = 1)
                res_df[f'BS_opt_t{t}_s{s}_put'] = res_df.apply(lambda x: BlackSholes(x['SnP500'], s, t/12, x[t]/100, x['VIX']/100, 'put'), axis = 1)
            else:
                raise Exception('Not yet implemented.')

    res_df = res_df.drop(columns = rates_df.columns[1:])
    return res_df

def get_BS_market_delta(Asset_df, volatility_df, rates_df, strikes, tenors, type = 'vanilla'):

    Asset_df, volatility_df, rates_df  = Asset_df.copy(), volatility_df.copy(), rates_df.copy()
    res_df = Asset_df.copy()
    res_df = res_df.merge(volatility_df, on = 'Date', how = 'left')
    res_df = res_df.merge(rates_df, on = 'Date', how = 'left')

    for t in tenors:
        for s in strikes:
            if type == 'vanilla':
                res_df[f'BS_opt_t{t}_s{s}_call'] = res_df.apply(lambda x: BlackSholesDelta(x['SnP500'], s, t/12, x[t]/100, x['VIX']/100, 'call'), axis = 1)
                res_df[f'BS_opt_t{t}_s{s}_put'] = res_df.apply(lambda x: BlackSholesDelta(x['SnP500'], s, t/12, x[t]/100, x['VIX']/100, 'put'), axis = 1)
            else:
                raise Exception('Not yet implemented.')

    res_df = res_df.drop(columns = rates_df.columns[1:])
    return res_df

In [ ]:
BlackScholesPut( S=4783, 
                 K=5000, 
                 T=0.5, 
                 r=0.079, 
                 sigma=12.55 / 100, 
                 q=0).price()

181.61986369931083

In [9]:
get_BS_market_prices(Asset_df = SnP_df, 
                    volatility_df = VIX_df, 
                    rates_df = df_rates, 
                    strikes = [500,1000,1500], 
                    tenors = [6,12,24], 
                    type = 'vanilla')

,Date,SnP500,VIX,BS_opt_t6_s500_call,BS_opt_t6_s500_put,BS_opt_t6_s1000_call,BS_opt_t6_s1000_put,BS_opt_t6_s1500_call,BS_opt_t6_s1500_put,BS_opt_t12_s500_call,...,BS_opt_t12_s1000_call,BS_opt_t12_s1000_put,BS_opt_t12_s1500_call,BS_opt_t12_s1500_put,BS_opt_t24_s500_call,BS_opt_t24_s500_put,BS_opt_t24_s1000_call,BS_opt_t24_s1000_put,BS_opt_t24_s1500_call,BS_opt_t24_s1500_put
0,1990-01-02,353.399994,17.24,0.093623,127.352639,2.510669e-14,607.918025,0.000000,1088.577034,1.774957,...,1.994301e-07,571.471941,-2.464025e-14,1033.907908,11.790760,85.571889,0.004784,500.967036,0.000002,9.281434e+02
1,1990-01-03,359.690002,18.19,0.224725,121.073582,7.404854e-14,601.387717,0.000000,1081.926576,2.838830,...,1.985776e-06,564.812059,9.426346e-13,1027.063087,15.249847,82.143332,0.014448,493.491421,0.000013,9.200605e+02
2,1990-01-04,358.760010,19.22,0.315364,122.190331,2.167720e-12,602.509944,0.000000,1083.144921,3.402257,...,8.710721e-06,566.019451,1.820175e-11,1028.409168,16.756936,84.751082,0.028830,494.777130,0.000052,9.215025e+02
3,1990-01-05,355.670013,20.11,0.359872,125.445010,1.525935e-11,605.840288,0.000000,1086.595439,3.650930,...,2.241251e-05,569.386937,1.290218e-10,1031.915378,17.349637,88.604514,0.044738,498.224507,0.000129,9.251048e+02
4,1990-01-08,352.200012,20.26,0.312137,128.795168,1.369556e-11,609.166074,0.000000,1089.849116,3.374945,...,2.109946e-05,572.671944,1.318352e-10,1035.107890,16.558586,91.283464,0.043036,501.692805,0.000129,9.285748e+02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8486,2023-12-22,4753.919922,13.72,4267.020246,0.000000,3.780121e+03,0.000000,3293.220893,0.000000,4277.448332,...,3.800977e+03,0.000000,3.324505e+03,0.000000,4295.214557,0.000000,3836.509191,0.000000,3377.803826,6.162576e-09
8487,2023-12-26,4758.859863,13.77,4271.887147,0.000000,3.784914e+03,0.000000,3297.941713,0.000000,4282.435918,...,3.806012e+03,0.000000,3.329588e+03,0.000000,4299.695563,0.000000,3840.531263,0.000000,3381.366963,7.218349e-09
8488,2023-12-27,4773.450195,13.02,4286.428779,0.000000,3.799407e+03,0.000000,3312.385946,0.000000,4296.835643,...,3.820221e+03,0.000000,3.343607e+03,0.000000,4313.734567,0.000000,3854.018939,0.000000,3394.303311,5.194614e-10
8489,2023-12-28,4786.439941,12.44,4299.467225,0.000000,3.812495e+03,0.000000,3325.521791,0.000000,4309.968352,...,3.833497e+03,0.000000,3.357025e+03,0.000000,4327.275641,0.000000,3868.111341,0.000000,3408.947041,4.438749e-11


In [10]:
deltas = get_BS_market_delta(Asset_df = SnP_df, 
                    volatility_df = VIX_df, 
                    rates_df = df_rates, 
                    strikes = [500,1000,1500], 
                    tenors = [1,2,3], 
                    type = 'vanilla')

In [11]:
deltas

,Date,SnP500,VIX,BS_opt_t1_s500_call,BS_opt_t1_s500_put,BS_opt_t1_s1000_call,BS_opt_t1_s1000_put,BS_opt_t1_s1500_call,BS_opt_t1_s1500_put,BS_opt_t2_s500_call,...,BS_opt_t2_s1000_call,BS_opt_t2_s1000_put,BS_opt_t2_s1500_call,BS_opt_t2_s1500_put,BS_opt_t3_s500_call,BS_opt_t3_s500_put,BS_opt_t3_s1000_call,BS_opt_t3_s1000_put,BS_opt_t3_s1500_call,BS_opt_t3_s1500_put
0,1990-01-02,353.399994,17.24,2.864597e-12,-1.0,0.0,-1.0,0.0,-1.0,6.408085e-07,...,0.0,-1.0,0.0,-1.0,0.000087,-0.999913,0.0,-1.0,0.0,-1.0
1,1990-01-03,359.690002,18.19,3.047855e-10,-1.0,0.0,-1.0,0.0,-1.0,6.859042e-06,...,0.0,-1.0,0.0,-1.0,0.000391,-0.999609,0.0,-1.0,0.0,-1.0
2,1990-01-04,358.760010,19.22,1.816502e-09,-1.0,0.0,-1.0,0.0,-1.0,1.705006e-05,...,0.0,-1.0,0.0,-1.0,0.000682,-0.999318,0.0,-1.0,0.0,-1.0
3,1990-01-05,355.670013,20.11,3.615515e-09,-1.0,0.0,-1.0,0.0,-1.0,2.426417e-05,...,0.0,-1.0,0.0,-1.0,0.000835,-0.999165,0.0,-1.0,0.0,-1.0
4,1990-01-08,352.200012,20.26,1.712233e-09,-1.0,0.0,-1.0,0.0,-1.0,1.659111e-05,...,0.0,-1.0,0.0,-1.0,0.000649,-0.999351,0.0,-1.0,0.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8486,2023-12-22,4753.919922,13.72,1.000000e+00,0.0,1.0,0.0,1.0,0.0,1.000000e+00,...,1.0,0.0,1.0,0.0,1.000000,0.000000,1.0,0.0,1.0,0.0
8487,2023-12-26,4758.859863,13.77,1.000000e+00,0.0,1.0,0.0,1.0,0.0,1.000000e+00,...,1.0,0.0,1.0,0.0,1.000000,0.000000,1.0,0.0,1.0,0.0
8488,2023-12-27,4773.450195,13.02,1.000000e+00,0.0,1.0,0.0,1.0,0.0,1.000000e+00,...,1.0,0.0,1.0,0.0,1.000000,0.000000,1.0,0.0,1.0,0.0
8489,2023-12-28,4786.439941,12.44,1.000000e+00,0.0,1.0,0.0,1.0,0.0,1.000000e+00,...,1.0,0.0,1.0,0.0,1.000000,0.000000,1.0,0.0,1.0,0.0


In [10]:
deltas.to_csv('/Users/artyomkraevskiy/Desktop/RPD paper/Trading_agent/delta_data.csv', index = None)